# UR5 73-episode SmolVLA training — A100 40 GB autotuned

This notebook trains the validated **absolute-action** dataset `ur5_cube_clean_73ep_v2` using the proven SmolVLA/LeRobot 0.4.3 stack. It copies data to local Colab storage, verifies the archive and dataset, benchmarks real training batches from 16 to 128, reserves GPU memory instead of forcing 100% allocation, preserves the earlier run's sample exposure, logs to Weights & Biases, syncs complete checkpoints to Drive, and packages the final model.

Expected Drive input: the uploaded folder `pHYSICAL UR5/datasets/lerobot/ur5_cube_clean_73ep_v2` (with `data`, `meta`, and `videos`), or alternatively a `.tar` plus `.sha256`. Select an **A100 GPU runtime**, then run cells from top to bottom.

In [ ]:
# 1. Verify the assigned runtime before importing torch.
import json, os, platform, subprocess, sys, time
from pathlib import Path

def output(command):
    return subprocess.run(command, text=True, capture_output=True, check=False).stdout.strip()

gpu_report = output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"])
print(gpu_report)
assert gpu_report, "No NVIDIA GPU assigned"
gpu_name, memory_mib, driver = [item.strip() for item in gpu_report.split(",", 2)]
assert "A100" in gpu_name, f"Expected an A100 runtime, received {gpu_name}"
assert int(memory_mib) >= 38000, f"Expected the 40 GB A100, received {memory_mib} MiB"
torch_already_imported = "torch" in sys.modules
if torch_already_imported:
    print("Torch is already imported; continuing in the existing prepared runtime. Skip the install cell if its version checks already passed.")
runtime_report = {"python": sys.version, "platform": platform.platform(), "gpu": gpu_name, "memory_mib": int(memory_mib), "driver": driver}
print(json.dumps(runtime_report, indent=2))

In [ ]:
# 2. Mount Drive and configure this dataset/run. Edit DRIVE_ROOT only if your shortcut differs.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/pHYSICAL UR5")
DATASET_NAME = "ur5_cube_clean_73ep_v2"
REPO_ID = f"local/{DATASET_NAME}"
TASK = "Pick up the orange cube and place it in the bowl"
DATASET_DRIVE_FOLDER = DRIVE_ROOT / f"datasets/lerobot/{DATASET_NAME}"
DATASET_ARCHIVE = DRIVE_ROOT / f"datasets/lerobot/{DATASET_NAME}.tar"
DATASET_CHECKSUM = Path(f"{DATASET_ARCHIVE}.sha256")
LOCAL_ARCHIVE = Path("/content") / DATASET_ARCHIVE.name
LOCAL_DATASET = Path("/content") / DATASET_NAME
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints/ur5_smolvla_trained"
REPORT_ROOT = DRIVE_ROOT / "reports/training"
DEPLOYMENT_ROOT = DRIVE_ROOT / "deployment/downloaded_checkpoints"
assert DRIVE_ROOT.is_dir(), f"Drive root missing: {DRIVE_ROOT}"
folder_upload = DATASET_DRIVE_FOLDER.is_dir()
archive_upload = DATASET_ARCHIVE.is_file() and DATASET_CHECKSUM.is_file()
assert folder_upload or archive_upload, (f"Dataset not found as folder {DATASET_DRIVE_FOLDER} or as verified archive {DATASET_ARCHIVE}")
if folder_upload:
    for required in ("data", "meta", "videos"):
        assert (DATASET_DRIVE_FOLDER / required).is_dir(), f"Incomplete uploaded dataset: missing {required}"
for directory in (CHECKPOINT_ROOT, REPORT_ROOT, DEPLOYMENT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
    probe = directory / ".write_test"; probe.write_text("ok"); probe.unlink()
print("Drive paths verified; input mode:", "folder" if folder_upload else "archive")

## Install the proven software stack
The versions match the successful previous training workflow and resulting checkpoints already used by the local inference server. Do not upgrade packages in the middle of a run.

In [ ]:
%pip install -q --upgrade pip
%pip install -q --index-url https://download.pytorch.org/whl/cu126 "torch==2.7.1" "torchvision==0.22.1" "torchaudio==2.7.1"
%pip install -q "lerobot[smolvla]==0.4.3" "av>=12" "wandb>=0.19" psutil matplotlib num2words

import importlib.metadata as md, torch, torchvision
assert torch.cuda.is_available()
assert torch.__version__.startswith("2.7.1")
assert torchvision.__version__.startswith("0.22.1")
assert torchvision.extension._has_ops()
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
runtime_report.update({"torch": torch.__version__, "torchvision": torchvision.__version__, "lerobot": md.version("lerobot"), "cuda": torch.version.cuda})
print(json.dumps(runtime_report, indent=2))

In [ ]:
# 3. Copy the uploaded folder, or verify/extract the archive, onto fast local disk.
import hashlib, shutil, tarfile
from pathlib import PurePosixPath

if LOCAL_DATASET.exists(): shutil.rmtree(LOCAL_DATASET)
if folder_upload:
    print("Copying uploaded LeRobot folder from Drive to local Colab storage ...")
    shutil.copytree(DATASET_DRIVE_FOLDER, LOCAL_DATASET)
else:
    expected_sha = DATASET_CHECKSUM.read_text().split()[0].lower()
    digest = hashlib.sha256()
    with DATASET_ARCHIVE.open("rb") as source, LOCAL_ARCHIVE.open("wb") as target:
        while chunk := source.read(16 * 1024 * 1024):
            digest.update(chunk); target.write(chunk)
    assert digest.hexdigest() == expected_sha, f"Dataset SHA-256 mismatch: {digest.hexdigest()}"
    with tarfile.open(LOCAL_ARCHIVE, "r:") as archive:
        members = archive.getmembers()
        for member in members:
            path = PurePosixPath(member.name)
            assert not path.is_absolute() and ".." not in path.parts and not member.issym() and not member.islnk(), f"Unsafe archive member: {member.name}"
        archive.extractall("/content", members=members, filter="data")
assert LOCAL_DATASET.is_dir(), f"Archive did not create {LOCAL_DATASET}"
for required in ("data", "meta", "videos"):
    assert (LOCAL_DATASET / required).is_dir(), f"Local dataset is incomplete: missing {required}"
print("Dataset copied to local storage:", LOCAL_DATASET)

In [ ]:
# 4. Validate LeRobot metadata and every state/action vector before training.
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset(REPO_ID, root=LOCAL_DATASET, video_backend="pyav")
features = dataset.meta.features
assert dataset.meta.total_episodes == 73, dataset.meta.total_episodes
assert features["observation.images.base"]["dtype"] == "video"
assert features["observation.images.wrist"]["dtype"] == "video"
assert tuple(features["observation.state"]["shape"]) == (7,)
assert tuple(features["action"]["shape"]) == (7,)
assert TASK in dataset.meta.tasks.index
for index in range(len(dataset)):
    row = dataset.hf_dataset[index]
    state = np.asarray(row["observation.state"], dtype=np.float32)
    action = np.asarray(row["action"], dtype=np.float32)
    assert state.shape == action.shape == (7,) and np.isfinite(state).all() and np.isfinite(action).all()
    assert min(abs(float(state[6])), abs(float(state[6]) - 1.0)) < 1e-5
    assert min(abs(float(action[6])), abs(float(action[6]) - 1.0)) < 1e-5
print({"episodes": dataset.meta.total_episodes, "frames": dataset.meta.total_frames, "task": TASK, "validation": "PASS"})

In [ ]:
# 5. Connect W&B. This opens a secure login prompt; do not paste keys into notebook source.
import wandb
wandb.login()
WANDB_PROJECT = "smolvla_ur5_cube_clean_73ep_v2_a100"
print("W&B project:", WANDB_PROJECT)

## A100 batch autotuning
Each candidate performs real forward/backward optimizer steps using the actual two-camera dataset. Selection uses examples/second, rejects OOM or other failures, keeps at least 12% VRAM free, and does not assume that filling all 40 GB is beneficial. The benchmark checkpoints are disabled and deleted.

In [ ]:
# 6. Benchmark stable batches and choose the fastest one within the memory reserve.
import re, statistics, subprocess, threading

BATCH_CANDIDATES = [16, 32, 48, 64, 96, 128]
BENCHMARK_STEPS = 12
NUM_WORKERS = max(4, min(12, (os.cpu_count() or 8) - 2))
GPU_MEMORY_LIMIT_MIB = int(runtime_report["memory_mib"] * 0.88)
BENCHMARK_ROOT = Path("/content/a100_batch_benchmarks")
if BENCHMARK_ROOT.exists(): shutil.rmtree(BENCHMARK_ROOT)
BENCHMARK_ROOT.mkdir()

def gpu_sample():
    text = output(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits"])
    memory, utilization = [int(float(value.strip())) for value in text.split(",")[:2]]
    return memory, utilization

def benchmark(batch_size):
    run_dir = BENCHMARK_ROOT / f"batch_{batch_size}"
    command = [
        "lerobot-train", "--policy.type=smolvla", "--policy.pretrained_path=lerobot/smolvla_base",
        "--policy.device=cuda", "--policy.use_amp=true", "--policy.push_to_hub=false",
        f"--dataset.repo_id={REPO_ID}", f"--dataset.root={LOCAL_DATASET}", "--dataset.video_backend=pyav",
        f"--batch_size={batch_size}", f"--num_workers={NUM_WORKERS}", "--seed=42",
        "--wandb.enable=false", "--save_checkpoint=false", "--eval_freq=0",
        f"--steps={BENCHMARK_STEPS}", "--log_freq=1", f"--output_dir={run_dir}", f"--job_name=a100_bench_b{batch_size}",
    ]
    samples, lines, stop = [], [], threading.Event()
    def monitor():
        while not stop.wait(0.25):
            try: samples.append(gpu_sample())
            except Exception: pass
    thread = threading.Thread(target=monitor, daemon=True); thread.start()
    started = time.time()
    process = subprocess.Popen(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for line in process.stdout:
        print(f"[batch {batch_size}] {line}", end="")
        lines.append(line)
    code = process.wait(); stop.set(); thread.join(timeout=2)
    text = "".join(lines)
    update_times = [float(value) for value in re.findall(r"updt_s:([0-9.]+)", text)]
    stable_times = update_times[len(update_times)//2:]
    peak_memory = max((item[0] for item in samples), default=0)
    utilization_samples = [item[1] for item in samples]
    median_util = statistics.median(utilization_samples) if utilization_samples else 0
    throughput = batch_size / statistics.median(stable_times) if stable_times and code == 0 else 0.0
    result = {"batch_size": batch_size, "success": code == 0, "peak_memory_mib": peak_memory, "median_gpu_utilization": median_util, "examples_per_second": throughput, "elapsed_s": time.time() - started}
    shutil.rmtree(run_dir, ignore_errors=True)
    return result

benchmark_results = []
for candidate in BATCH_CANDIDATES:
    result = benchmark(candidate); benchmark_results.append(result); print(json.dumps(result, indent=2))
    if not result["success"]: break
eligible = [item for item in benchmark_results if item["success"] and item["peak_memory_mib"] <= GPU_MEMORY_LIMIT_MIB]
assert eligible, f"No A100 batch candidate succeeded: {benchmark_results}"
selected = max(eligible, key=lambda item: item["examples_per_second"])
BATCH_SIZE = selected["batch_size"]
print("SELECTED BATCH", BATCH_SIZE, json.dumps(selected, indent=2))

In [ ]:
# 7. Preserve the previous successful run's sample exposure instead of blindly keeping 60k steps.
import math, random
TARGET_EPOCHS = 8.0
TRAIN_STEPS = int(math.ceil((dataset.meta.total_frames * TARGET_EPOCHS / BATCH_SIZE) / 500.0) * 500)
SAVE_FREQ = max(500, int(math.ceil((dataset.meta.total_frames / BATCH_SIZE) / 500.0) * 500))
WARMUP_STEPS = min(1000, max(200, TRAIN_STEPS // 20))
SEED = 42
RUN_NAME = f"{DATASET_NAME}_smolvla_a100_b{BATCH_SIZE}_{TRAIN_STEPS}_steps"
TRAIN_OUTPUT = Path("/content/outputs") / RUN_NAME
LOCAL_REPORT_DIR = Path("/content/training_reports") / RUN_NAME
DRIVE_RUN_DIR = CHECKPOINT_ROOT / RUN_NAME
DRIVE_CHECKPOINTS_DIR = DRIVE_RUN_DIR / "checkpoints"
assert not TRAIN_OUTPUT.exists() and not DRIVE_RUN_DIR.exists(), "Run name already exists; change RUN_NAME or use a fresh runtime"
LOCAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
TRAIN_ARGS = [
    "lerobot-train", "--policy.type=smolvla", "--policy.pretrained_path=lerobot/smolvla_base",
    "--policy.device=cuda", "--policy.use_amp=true", "--policy.push_to_hub=false",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}", f"--policy.scheduler_decay_steps={TRAIN_STEPS}",
    f"--dataset.repo_id={REPO_ID}", f"--dataset.root={LOCAL_DATASET}", "--dataset.video_backend=pyav",
    f"--batch_size={BATCH_SIZE}", f"--num_workers={NUM_WORKERS}", f"--seed={SEED}",
    "--wandb.enable=true", f"--wandb.project={WANDB_PROJECT}",
    "--save_checkpoint=true", "--eval_freq=0", f"--steps={TRAIN_STEPS}", f"--save_freq={SAVE_FREQ}",
    "--log_freq=10", f"--output_dir={TRAIN_OUTPUT}", f"--job_name={RUN_NAME}",
]
resolved = {"dataset": DATASET_NAME, "episodes": dataset.meta.total_episodes, "frames": dataset.meta.total_frames, "target_epochs": TARGET_EPOCHS, "batch_size": BATCH_SIZE, "steps": TRAIN_STEPS, "save_freq": SAVE_FREQ, "workers": NUM_WORKERS, "warmup_steps": WARMUP_STEPS, "action_mode": "absolute", "benchmark_results": benchmark_results, "command": TRAIN_ARGS}
(LOCAL_REPORT_DIR / "resolved_training_config.json").write_text(json.dumps(resolved, indent=2))
print(json.dumps(resolved, indent=2))

## Train, monitor, and sync
Training runs on `/content`, not mounted Drive. A monitor samples GPU utilization and VRAM every two seconds. Only complete checkpoints are copied atomically to Drive. The console and W&B both show live progress.

In [ ]:
# 8. Start the final run. Leave this cell running until completion.
import csv
TRAIN_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
TRAIN_LOG = LOCAL_REPORT_DIR / "training.log"
GPU_LOG = LOCAL_REPORT_DIR / "gpu_metrics.csv"
DRIVE_RUN_DIR.mkdir(parents=True); DRIVE_CHECKPOINTS_DIR.mkdir()
shutil.copy2(LOCAL_REPORT_DIR / "resolved_training_config.json", DRIVE_RUN_DIR / "resolved_training_config.json")

def checkpoint_complete(path):
    required = [path / "pretrained_model/model.safetensors", path / "pretrained_model/config.json", path / "training_state/training_step.json"]
    return path.is_dir() and all(item.is_file() for item in required)

def sync_checkpoints():
    checkpoint_dir = TRAIN_OUTPUT / "checkpoints"
    if not checkpoint_dir.exists(): return
    for source in sorted(checkpoint_dir.iterdir()):
        if source.name == "last" or not checkpoint_complete(source): continue
        target = DRIVE_CHECKPOINTS_DIR / source.name
        if target.exists(): continue
        staging = DRIVE_CHECKPOINTS_DIR / f".{source.name}.copying"
        shutil.rmtree(staging, ignore_errors=True); shutil.copytree(source, staging); staging.rename(target)
        print("SYNCED COMPLETE CHECKPOINT:", target)

monitor_stop = threading.Event()
def gpu_monitor():
    with GPU_LOG.open("w", newline="") as stream:
        writer = csv.writer(stream); writer.writerow(["time_s", "memory_mib", "utilization_percent"]); started = time.time()
        while not monitor_stop.wait(2.0):
            try:
                memory, utilization = gpu_sample(); writer.writerow([time.time() - started, memory, utilization]); stream.flush()
            except Exception: pass
monitor_thread = threading.Thread(target=gpu_monitor, daemon=True); monitor_thread.start()

line_count = 0
with TRAIN_LOG.open("w") as log:
    process = subprocess.Popen(TRAIN_ARGS, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    for line in process.stdout:
        print(line, end=""); log.write(line); log.flush(); line_count += 1
        sync_checkpoints()
        if line_count % 100 == 0:
            shutil.copy2(TRAIN_LOG, DRIVE_RUN_DIR / "training_in_progress.log")
            shutil.copy2(GPU_LOG, DRIVE_RUN_DIR / "gpu_metrics_in_progress.csv")
    return_code = process.wait()
monitor_stop.set(); monitor_thread.join(timeout=3); sync_checkpoints()
shutil.copy2(TRAIN_LOG, DRIVE_RUN_DIR / "training.log")
shutil.copy2(GPU_LOG, DRIVE_RUN_DIR / "gpu_metrics.csv")
assert return_code == 0, f"Training failed with return code {return_code}; inspect {TRAIN_LOG}"
print("TRAINING COMPLETE", RUN_NAME)

In [ ]:
# 9. Plot actual A100 utilization and copy the final complete model.
import matplotlib.pyplot as plt, pandas as pd
metrics = pd.read_csv(GPU_LOG)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(metrics.time_s / 60, metrics.utilization_percent); axes[0].set_ylabel("GPU utilization %"); axes[0].set_ylim(0, 105); axes[0].grid()
axes[1].plot(metrics.time_s / 60, metrics.memory_mib / 1024); axes[1].set_ylabel("VRAM GiB"); axes[1].set_xlabel("Minutes"); axes[1].grid()
plt.tight_layout(); plt.show()
summary = {"median_gpu_utilization": float(metrics.utilization_percent.median()), "p95_gpu_utilization": float(metrics.utilization_percent.quantile(0.95)), "peak_vram_gib": float(metrics.memory_mib.max() / 1024)}
print(json.dumps(summary, indent=2))

models = sorted(TRAIN_OUTPUT.glob("checkpoints/*/pretrained_model"), key=lambda path: int(path.parent.name) if path.parent.name.isdigit() else -1)
assert models, "No complete local model checkpoint found"
FINAL_MODEL = models[-1]
final_target = DRIVE_RUN_DIR / "pretrained_model"
staging = DRIVE_RUN_DIR / ".pretrained_model.copying"
shutil.rmtree(staging, ignore_errors=True); shutil.copytree(FINAL_MODEL, staging)
assert (staging / "model.safetensors").is_file() and (staging / "config.json").is_file()
staging.rename(final_target)
(DRIVE_RUN_DIR / "gpu_summary.json").write_text(json.dumps(summary, indent=2))
print("FINAL MODEL:", final_target)

In [ ]:
# 10. Package the complete pretrained_model for downloading to the robot PC.
export_archive = DEPLOYMENT_ROOT / f"{RUN_NAME}_complete_checkpoint.tar"
with tarfile.open(export_archive, "w") as archive:
    archive.add(DRIVE_RUN_DIR / "pretrained_model", arcname="pretrained_model")
digest = hashlib.sha256()
with export_archive.open("rb") as stream:
    while chunk := stream.read(16 * 1024 * 1024): digest.update(chunk)
checksum = Path(f"{export_archive}.sha256")
checksum.write_text(f"{digest.hexdigest()}  {export_archive.name}\n")
shutil.copy2(LOCAL_REPORT_DIR / "resolved_training_config.json", REPORT_ROOT / f"{RUN_NAME}_config.json")
shutil.copy2(TRAIN_LOG, REPORT_ROOT / f"{RUN_NAME}_training.log")
shutil.copy2(GPU_LOG, REPORT_ROOT / f"{RUN_NAME}_gpu_metrics.csv")
print("READY TO DOWNLOAD:", export_archive)
print("SHA-256:", digest.hexdigest())

## Finished
The final weights, all complete checkpoints, W&B run, training log, resolved configuration, and GPU-utilization trace are preserved in Drive. Do not execute the checkpoint physically until it passes offline and shadow validation.